# Define Transfer Routes - Blue Line 

Identify blue line records in xfer routes for each record.

In [1]:
import warnings
warnings.simplefilter("always")

### Read Inputs

In [2]:
## Data Preparation and Summary
import pandas as pd
import numpy as np

# Set directory
data_dir = "../input/"
interim_dir = '../output/interim/'
output_dir = '../output/'

df_full = pd.read_excel(data_dir + "od_20240422_sandagca_weighted_combined_draftfinal.xlsx", sheet_name="OD_RESULTS", engine='openpyxl') # raw survey

In [3]:
df_full.shape

(35260, 201)

In [4]:
df_weight = pd.read_excel(data_dir + "2023OBS_ReWeighted_2024-05-24.xlsx",
                          sheet_name="Sheet1", engine='openpyxl')
df_weight.shape

(35260, 8)

In [5]:
df = pd.merge(df_full, df_weight[['ID', 'REWEIGHTED_LINKED', 'REWEIGHTED_UNLINKED']],
              on='ID', how='left')
df.shape

(35260, 203)

### Filter Blue Line Xfer Records

In [6]:
# xfer columns in survey
prev_xfer_cols = ['TRIP_FIRST_ROUTE[Code]','TRIP_SECOND_ROUTE[Code]','TRIP_THIRD_ROUTE[Code]','TRIP_FOURTH_ROUTE[Code]']
next_xfer_cols = ['TRIP_NEXT_ROUTE[Code]','TRIP_AFTER_ROUTE[Code]','TRIP_3RD_ROUTE[Code]','TRIP_LAST4TH_RTE[Code]']

In [7]:
# find all survey records that Blue Line exist in transfer columns
df_xfer_blue = df[df[prev_xfer_cols+next_xfer_cols].eq('MTS_1_Blue').any(axis=1)][['ID']+prev_xfer_cols+['ROUTE_DIRECTION[Code]']+next_xfer_cols+['REWEIGHTED_LINKED']]
df_xfer_blue

,ID,TRIP_FIRST_ROUTE[Code],TRIP_SECOND_ROUTE[Code],TRIP_THIRD_ROUTE[Code],TRIP_FOURTH_ROUTE[Code],ROUTE_DIRECTION[Code],TRIP_NEXT_ROUTE[Code],TRIP_AFTER_ROUTE[Code],TRIP_3RD_ROUTE[Code],TRIP_LAST4TH_RTE[Code],REWEIGHTED_LINKED
3,31644,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,15.506118
10,31758,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,3.047902
11,31772,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,3.047902
28,29182,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,4.077759
42,29385,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,4.077759
...,...,...,...,...,...,...,...,...,...,...,...
35207,71467,MTS_1_Blue,NaN,NaN,NaN,MTS_1_Orange_01,NaN,NaN,NaN,NaN,4.240665
35223,20736,NaN,NaN,NaN,NaN,MTS_1_Orange_01,MTS_1_Blue,NaN,NaN,NaN,8.984289
35235,30499,NaN,NaN,NaN,NaN,MTS_1_Orange_01,MTS_1_Blue,NaN,NaN,NaN,4.250952
35238,33540,NaN,NaN,NaN,NaN,MTS_1_Orange_01,MTS_1_Blue,NaN,NaN,NaN,17.322049


In [8]:
def find_position(row, cols):
    for col in cols:
        if row[col] == 'MTS_1_Blue':
            return col
    return np.nan

# Apply to both sets
df_xfer_blue['prev_position'] = df_xfer_blue.apply(lambda row: find_position(row, prev_xfer_cols), axis=1)
df_xfer_blue['next_position'] = df_xfer_blue.apply(lambda row: find_position(row, next_xfer_cols), axis=1)

df_xfer_blue['xfer_position'] = df_xfer_blue['prev_position'].fillna(df_xfer_blue['next_position'])

In [9]:
df_xfer_blue.head()

,ID,TRIP_FIRST_ROUTE[Code],TRIP_SECOND_ROUTE[Code],TRIP_THIRD_ROUTE[Code],TRIP_FOURTH_ROUTE[Code],ROUTE_DIRECTION[Code],TRIP_NEXT_ROUTE[Code],TRIP_AFTER_ROUTE[Code],TRIP_3RD_ROUTE[Code],TRIP_LAST4TH_RTE[Code],REWEIGHTED_LINKED,prev_position,next_position,xfer_position
3,31644,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,15.506118,TRIP_FIRST_ROUTE[Code],NaN,TRIP_FIRST_ROUTE[Code]
10,31758,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,3.047902,TRIP_FIRST_ROUTE[Code],NaN,TRIP_FIRST_ROUTE[Code]
11,31772,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,3.047902,TRIP_FIRST_ROUTE[Code],NaN,TRIP_FIRST_ROUTE[Code]
28,29182,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,4.077759,TRIP_FIRST_ROUTE[Code],NaN,TRIP_FIRST_ROUTE[Code]
42,29385,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,4.077759,TRIP_FIRST_ROUTE[Code],NaN,TRIP_FIRST_ROUTE[Code]


In [10]:
# list all xfer station coordinates
stn_coord_cols = ['PREV_TRAN_1_ON_BUS [LAT]',
                'PREV_TRAN_1_ON_BUS [LONG]',
                'PREV_TRAN_1_OFF_BUS [LAT]',
                'PREV_TRAN_1_OFF_BUS [LONG]',
                'PREV_TRAN_2_ON_BUS [LAT]',
                'PREV_TRAN_2_ON_BUS [LONG]',
                'PREV_TRAN_2_OFF_BUS [LAT]',
                'PREV_TRAN_2_OFF_BUS [LONG]',
                'PREV_TRAN_3_ON_BUS [LAT]',
                'PREV_TRAN_3_ON_BUS [LONG]',
                'PREV_TRAN_3_OFF_BUS [LAT]',
                'PREV_TRAN_3_OFF_BUS [LONG]',
                'PREV_TRAN_4_ON_BUS [LAT]',
                'PREV_TRAN_4_ON_BUS [LONG]',
                'PREV_TRAN_4_OFF_BUS [LAT]',
                'PREV_TRAN_4_OFF_BUS [LONG]',
                'NEXT_TRAN_1_ON_BUS [LAT]',
                'NEXT_TRAN_1_ON_BUS [LONG]',
                'NEXT_TRAN_1_OFF_BUS [LAT]',
                'NEXT_TRAN_1_OFF_BUS [LONG]',
                'NEXT_TRAN_2_ON_BUS [LAT]',
                'NEXT_TRAN_2_ON_BUS [LONG]',
                'NEXT_TRAN_2_OFF_BUS [LAT]',
                'NEXT_TRAN_2_OFF_BUS [LONG]',
                'NEXT_TRAN_3_ON_BUS [LAT]',
                'NEXT_TRAN_3_ON_BUS [LONG]',
                'NEXT_TRAN_3_OFF_BUS [LAT]',
                'NEXT_TRAN_3_OFF_BUS [LONG]',
                'NEXT_TRAN_4_ON_BUS [LAT]',
                'NEXT_TRAN_4_ON_BUS [LONG]',
                'NEXT_TRAN_4_OFF_BUS [LAT]',
                'NEXT_TRAN_4_OFF_BUS [LONG]']

In [11]:
df_xfer_blue = df_xfer_blue.merge(df[['ID']+stn_coord_cols+['STOP_ON [ADDR]','STOP_ON [LAT]','STOP_ON [LONG]','STOP_OFF [ADDR]','STOP_OFF [LAT]','STOP_OFF [LONG]']],on='ID',how='left')
df_xfer_blue.shape

(4396, 52)

In [12]:
# assign corresponding station coordinates based on the transfer routes position
xfer_pos = [
    df_xfer_blue['xfer_position'] == 'TRIP_FIRST_ROUTE[Code]',
    df_xfer_blue['xfer_position'] == 'TRIP_SECOND_ROUTE[Code]',
    df_xfer_blue['xfer_position'] == 'TRIP_THIRD_ROUTE[Code]',
    df_xfer_blue['xfer_position'] == 'TRIP_FOURTH_ROUTE[Code]',
    
    df_xfer_blue['xfer_position'] == 'TRIP_NEXT_ROUTE[Code]',
    df_xfer_blue['xfer_position'] == 'TRIP_AFTER_ROUTE[Code]',
    df_xfer_blue['xfer_position'] == 'TRIP_3RD_ROUTE[Code]',
    df_xfer_blue['xfer_position'] == 'TRIP_LAST4TH_RTE[Code]'
]
xfer_on_lat = [
    df_xfer_blue['PREV_TRAN_1_ON_BUS [LAT]'],
    df_xfer_blue['PREV_TRAN_2_ON_BUS [LAT]'],
    df_xfer_blue['PREV_TRAN_3_ON_BUS [LAT]'],
    df_xfer_blue['PREV_TRAN_4_ON_BUS [LAT]'],

    df_xfer_blue['NEXT_TRAN_1_ON_BUS [LAT]'],
    df_xfer_blue['NEXT_TRAN_2_ON_BUS [LAT]'],
    df_xfer_blue['NEXT_TRAN_3_ON_BUS [LAT]'],
    df_xfer_blue['NEXT_TRAN_4_ON_BUS [LAT]']
]
xfer_on_long = [
    df_xfer_blue['PREV_TRAN_1_ON_BUS [LONG]'],
    df_xfer_blue['PREV_TRAN_2_ON_BUS [LONG]'],
    df_xfer_blue['PREV_TRAN_3_ON_BUS [LONG]'],
    df_xfer_blue['PREV_TRAN_4_ON_BUS [LONG]'],

    df_xfer_blue['NEXT_TRAN_1_ON_BUS [LONG]'],
    df_xfer_blue['NEXT_TRAN_2_ON_BUS [LONG]'],
    df_xfer_blue['NEXT_TRAN_3_ON_BUS [LONG]'],
    df_xfer_blue['NEXT_TRAN_4_ON_BUS [LONG]']
]

xfer_off_lat = [
    df_xfer_blue['PREV_TRAN_1_OFF_BUS [LAT]'],
    df_xfer_blue['PREV_TRAN_2_OFF_BUS [LAT]'],
    df_xfer_blue['PREV_TRAN_3_OFF_BUS [LAT]'],
    df_xfer_blue['PREV_TRAN_4_OFF_BUS [LAT]'],

    df_xfer_blue['NEXT_TRAN_1_OFF_BUS [LAT]'],
    df_xfer_blue['NEXT_TRAN_2_OFF_BUS [LAT]'],
    df_xfer_blue['NEXT_TRAN_3_OFF_BUS [LAT]'],
    df_xfer_blue['NEXT_TRAN_4_OFF_BUS [LAT]']
]
xfer_off_long = [
    df_xfer_blue['PREV_TRAN_1_OFF_BUS [LONG]'],
    df_xfer_blue['PREV_TRAN_2_OFF_BUS [LONG]'],
    df_xfer_blue['PREV_TRAN_3_OFF_BUS [LONG]'],
    df_xfer_blue['PREV_TRAN_4_OFF_BUS [LONG]'],

    df_xfer_blue['NEXT_TRAN_1_OFF_BUS [LONG]'],
    df_xfer_blue['NEXT_TRAN_2_OFF_BUS [LONG]'],
    df_xfer_blue['NEXT_TRAN_3_OFF_BUS [LONG]'],
    df_xfer_blue['NEXT_TRAN_4_OFF_BUS [LONG]']
]

df_xfer_blue['xfer_on_lat'] = np.select(xfer_pos, xfer_on_lat, default=None)
df_xfer_blue['xfer_on_long'] = np.select(xfer_pos, xfer_on_long, default=None)
df_xfer_blue['xfer_off_lat'] = np.select(xfer_pos, xfer_off_lat, default=None)
df_xfer_blue['xfer_off_long'] = np.select(xfer_pos, xfer_off_long, default=None)

In [13]:
df_xfer_blue.head()

,ID,TRIP_FIRST_ROUTE[Code],TRIP_SECOND_ROUTE[Code],TRIP_THIRD_ROUTE[Code],TRIP_FOURTH_ROUTE[Code],ROUTE_DIRECTION[Code],TRIP_NEXT_ROUTE[Code],TRIP_AFTER_ROUTE[Code],TRIP_3RD_ROUTE[Code],TRIP_LAST4TH_RTE[Code],...,STOP_ON [ADDR],STOP_ON [LAT],STOP_ON [LONG],STOP_OFF [ADDR],STOP_OFF [LAT],STOP_OFF [LONG],xfer_on_lat,xfer_on_long,xfer_off_lat,xfer_off_long
0,31644,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,UTC Transit Center,32.868324,-117.213439,Carlsbad Village Station Stall 4,33.160922,-117.351168,32.71106,-117.153721,32.869204,-117.214011
1,31758,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,UTC Transit Center,32.868324,-117.213439,N Torrey Pines Rd & Torrey Pines Golf Course,32.905431,-117.243229,32.544568,-117.029552,32.869204,-117.214011
2,31772,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,UTC Transit Center,32.868324,-117.213439,Camino Del Mar & 24th St,32.968910,-117.267619,32.769914,-117.205062,32.869204,-117.214011
3,29182,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,UTC Transit Center,32.868324,-117.213439,N Torrey Pines Rd & Torrey Pines Golf Course,32.905431,-117.243229,32.716386,-117.168811,32.869204,-117.214011
4,29385,MTS_1_Blue,NaN,NaN,NaN,NCT_2_101_00,NaN,NaN,NaN,NaN,...,Nobel Dr & La Jolla Village Square Drwy,32.868011,-117.231658,N Torrey Pines Rd & Torrey Pines Golf Course,32.905431,-117.243229,32.721494,-117.169898,32.866862,-117.230462


In [14]:
df_xfer_blue.drop(columns=stn_coord_cols).to_csv(interim_dir + 'survey_blue_line_xfer_routes_on_off_coords.csv',index=False)

QAQC

In [15]:
prev_xfer_cols = ['TRIP_FIRST_ROUTE[Code]','TRIP_SECOND_ROUTE[Code]','TRIP_THIRD_ROUTE[Code]','TRIP_FOURTH_ROUTE[Code]']
next_xfer_cols = ['TRIP_NEXT_ROUTE[Code]','TRIP_AFTER_ROUTE[Code]','TRIP_3RD_ROUTE[Code]','TRIP_LAST4TH_RTE[Code]']

In [ ]:
# Check: for each row, only one xfer columns has 'MTS_1_Blue' --> no multiple transfers on Blue Line
df_xfer_blue[prev_xfer_cols+next_xfer_cols].eq('MTS_1_Blue').sum(axis=1).unique()

array([1])

In [ ]:
# Check: Random check the defination of prev_postion or next_position
df_xfer_blue[df_xfer_blue['prev_position']=='TRIP_FIRST_ROUTE[Code]']['TRIP_FIRST_ROUTE[Code]'].unique()
df_xfer_blue[df_xfer_blue['next_position']=='TRIP_AFTER_ROUTE[Code]']['TRIP_AFTER_ROUTE[Code]'].unique()

df_xfer_blue[df_xfer_blue['xfer_position']=='TRIP_AFTER_ROUTE[Code]']['TRIP_AFTER_ROUTE[Code]'].unique()

array(['MTS_1_Blue'], dtype=object)